In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("data (2).csv")

# Remove unnecessary columns
df.drop(["id", "Unnamed: 32"], axis=1, inplace=True)

# ------------------------------
# Encode Target
# M = Malignant = 1
# B = Benign = 0
# ------------------------------

encoder = LabelEncoder()
df["diagnosis"] = encoder.fit_transform(df["diagnosis"])

In [3]:
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

In [5]:
selector = SelectKBest(score_func=f_classif, k=15)

X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("\nTop Selected Features:\n")
for feature in selected_features:
    print(feature)



Top Selected Features:

radius_mean
perimeter_mean
area_mean
compactness_mean
concavity_mean
concave points_mean
radius_se
perimeter_se
area_se
radius_worst
perimeter_worst
area_worst
compactness_worst
concavity_worst
concave points_worst


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [8]:
models = {
    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            random_state=42
        )
}

In [9]:
best_model = None
best_accuracy = 0

print("\nMODEL COMPARISON\n")

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    print(f"{name}: {accuracy:.4f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model

print("\nBest Model Selected Successfully!")
print("Accuracy:", round(best_accuracy*100, 2), "%")



MODEL COMPARISON

Logistic Regression: 0.9737
Random Forest: 0.9561
Gradient Boosting: 0.9561

Best Model Selected Successfully!
Accuracy: 97.37 %


In [10]:
final_predictions = best_model.predict(X_test)

print("\nClassification Report:\n")
print(classification_report(y_test, final_predictions))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, final_predictions))



Classification Report:

              precision    recall  f1-score   support

           0       0.96      1.00      0.98        72
           1       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114


Confusion Matrix:

[[72  0]
 [ 3 39]]


In [11]:
sample_patient = X.iloc[0].values.reshape(1, -1)

sample_patient = selector.transform(sample_patient)

sample_patient = scaler.transform(sample_patient)

prediction = best_model.predict(sample_patient)

if prediction[0] == 1:
    print("\nPrediction: MALIGNANT (Cancer Detected)")
else:
    print("\nPrediction: BENIGN (No Cancer)")


Prediction: MALIGNANT (Cancer Detected)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SelectKBest was fitted with feature names
  warnings.warn(
